In [8]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
import MEArec as mr
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques
)

# 用于PSTH计算的导入
import neo
from elephant.kernels import GaussianKernel
from elephant.statistics import instantaneous_rate
from quantities import ms

In [2]:
target_month_session_names = [
    'mouse6_021322_natural_image_001',  # 第1个月 (Session 1)
    'mouse6_022522_natural_image_001',  # 第2个月 (Session 3)
    'mouse6_031722_natural_image_001',  # 第3个月 (Session 4)
    'mouse6_042422_natural_image_001',  # 第4个月 (Session 7)
    'mouse6_052422_natural_image_001',  # 第5个月 (Session 8)
    'mouse6_062422_natural_image_001',  # 第6个月 (Session 9)
    'mouse6_072322_natural_image_001',  # 第7个月 (Session 10)
    'mouse6_082322_natural_image_001',  # 第8个月 (Session 11)
    'mouse6_092422_natural_image_001',  # 第9个月 (Session 12)
    'mouse6_102122_natural_image_001',  # 第10个月 (Session 13)
    'mouse6_112022_natural_image_001',  # 第11个月 (Session 14)
    'mouse6_122022_natural_image_001',  # 第12个月 (Session 15)
]

In [5]:
trigger_df = pd.read_csv("/media/ubuntu/sda/data/mouse6/output/01_get_trigger/trigger_time.tsv", sep = '\t', index_col= 0)

In [48]:
# 配置路径和参数
BASE_DIR = "/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/"
CLIQUE_DIR = os.path.join(BASE_DIR, "clique_0")
OUTPUT_DIR = "/media/ubuntu/sda/mouse_test/processed_results/psth_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# PSTH参数设置
SAMPLING_RATE = 10000  # Hz，目标采样率
EXTEND_TIME = 0.25  # 秒，左右延长时间
STIMULUS_DURATION = 1  # 秒，刺激持续时间
bin_size_ms = 50  # 10ms的bin size
bin_size_s = bin_size_ms / 1000.0
gk = GaussianKernel(150 * ms)  # 25ms的Gaussian kernel

# 计算时间轴（使用extended时间窗）
total_time_extended = EXTEND_TIME + STIMULUS_DURATION + EXTEND_TIME
time_bins = np.arange(0, total_time_extended, bin_size_s)
n_time_bins = len(time_bins)

print(f"PSTH参数:")
print(f"  Total time: {total_time_extended} s")
print(f"  Bin size: {bin_size_ms} ms")
print(f"  Number of time bins: {n_time_bins}")

# 读取第一个session (021322) 的neuron信息作为基准
baseline_session = 'mouse6_021322_natural_image_001'
baseline_session_dir = os.path.join(CLIQUE_DIR, baseline_session)
baseline_neuron_inf_path = os.path.join(baseline_session_dir, "neuron_inf.pickle")
baseline_gt_detect_path = os.path.join(baseline_session_dir, "gt_detect_array.csv")

print(f"\n读取基准session: {baseline_session}")
with open(baseline_neuron_inf_path, 'rb') as f:
    baseline_neuron_inf = pickle.load(f)

# 读取baseline的spike数据
baseline_spike_data = {}
if os.path.exists(baseline_gt_detect_path):
    baseline_gt_detect_df = pd.read_csv(baseline_gt_detect_path)
    baseline_gt_detect_df['time'] = pd.to_numeric(baseline_gt_detect_df['time'], errors='coerce')
    baseline_gt_detect_df['unit_id'] = pd.to_numeric(baseline_gt_detect_df['unit_id'], errors='coerce')
    baseline_gt_detect_df = baseline_gt_detect_df.dropna(subset=['time', 'unit_id'])
    
    for neuron_id in baseline_neuron_inf.keys():
        neuron_spikes = baseline_gt_detect_df[baseline_gt_detect_df['unit_id'] == neuron_id]['time'].values
        baseline_spike_data[neuron_id] = neuron_spikes

# 获取基准neuron列表（按neuron_id排序）
baseline_neuron_ids = sorted(baseline_neuron_inf.keys())
n_neurons = len(baseline_neuron_ids)
print(f"基准session有 {n_neurons} 个neurons")

# 从session名称中提取date（例如：mouse6_021322_natural_image_001 -> 021322）
def extract_date_from_session(session_name):
    parts = session_name.split('_')
    if len(parts) >= 2:
        return parts[1]
    return None

# ============================================================================
# Step 1: 计算correlation和识别outlier，过滤trigger_df
# ============================================================================
print(f"\n{'='*60}")
print("Step 1: 计算correlation和识别outlier")
print(f"{'='*60}")

from scipy.stats import pearsonr

# 存储过滤后的trigger_df（去除outlier后）
filtered_trigger_df_dict = {}

# 遍历所有session，计算correlation并识别outlier
for session_name in target_month_session_names:
    session_date = extract_date_from_session(session_name)
    if session_date is None:
        continue
    
    # 将date转换为trigger_df中的格式
    trigger_date = int(session_date)
    session_trigger_df = trigger_df[trigger_df['date'] == trigger_date].copy()
    
    if len(session_trigger_df) == 0:
        continue
    
    
    # 读取该session的spike数据
    session_dir = os.path.join(CLIQUE_DIR, session_name)
    session_gt_detect_path = os.path.join(session_dir, "gt_detect_array.csv")
    
    if not os.path.exists(session_gt_detect_path):
        print(f"  警告: 未找到gt_detect_array.csv，跳过")
        continue
    
    # 读取gt_detect_array
    gt_detect_df = pd.read_csv(session_gt_detect_path)
    gt_detect_df['time'] = pd.to_numeric(gt_detect_df['time'], errors='coerce')
    gt_detect_df['unit_id'] = pd.to_numeric(gt_detect_df['unit_id'], errors='coerce')
    gt_detect_df = gt_detect_df.dropna(subset=['time', 'unit_id'])
    
    # 为每个image计算correlation和识别outlier
    outlier_orders = set()  # 存储需要去除的outlier trial的order
    
    for image in session_trigger_df['image'].unique():
        image_trigger_df = session_trigger_df[session_trigger_df['image'] == image].copy()
        
        # 初始化该image的DataFrame，行索引为baseline_neuron_ids
        image_firing_rate_df = pd.DataFrame(index=baseline_neuron_ids)
        
        # 按order排序
        image_trigger_df = image_trigger_df.sort_values('order')
        
        # 遍历每个trial（按order），计算firing rate
        for _, trial in image_trigger_df.iterrows():
            start_time = int(trial['start'])
            end_time = int(trial['end'])
            
            # 获取该trial内的spikes
            trial_spikes = gt_detect_df[
                (gt_detect_df['time'] >= start_time) & 
                (gt_detect_df['time'] < end_time)
            ]
            
            # 计算每个neuron的spike count
            neuron_counts = trial_spikes['unit_id'].value_counts()
            
            # 创建该trial的firing rate向量（按baseline_neuron_ids对齐）
            trial_firing_rate = pd.Series(index=baseline_neuron_ids, dtype=float)
            for neuron_id in baseline_neuron_ids:
                if neuron_id in neuron_counts:
                    # 计算firing rate (spikes per second)
                    trial_duration = (end_time - start_time) / SAMPLING_RATE  # 秒
                    trial_firing_rate[neuron_id] = neuron_counts[neuron_id] / trial_duration
                else:
                    trial_firing_rate[neuron_id] = 0.0
            
            # 添加到DataFrame中（列名为order）
            image_firing_rate_df[trial['order']] = trial_firing_rate
        
        # 填充NaN为0
        image_firing_rate_df = image_firing_rate_df.fillna(0)
        
        # 计算correlation矩阵
        num_trials = image_firing_rate_df.shape[1]
        correlation_matrix = np.zeros((num_trials, num_trials))
        
        for i in range(num_trials):
            for j in range(num_trials):
                trial_i = image_firing_rate_df.iloc[:, i].values
                trial_j = image_firing_rate_df.iloc[:, j].values
                correlation_matrix[i, j], _ = pearsonr(trial_i, trial_j)
        

        mean_correlations = (correlation_matrix.sum(axis=0) - 1) / (num_trials - 1)
        
        # 如果平均correlation <= 0.6，则认为是outlier
        outlier_threshold = 0.6
        trial_orders = image_trigger_df['order'].values
        outlier_indices = np.where(mean_correlations <= outlier_threshold)[0]
        image_outlier_orders = trial_orders[outlier_indices]
        outlier_orders.update(image_outlier_orders)
            
    # 过滤trigger_df，去除outlier trials
    filtered_session_trigger_df = session_trigger_df[~session_trigger_df['order'].isin(outlier_orders)].copy()
    filtered_trigger_df_dict[session_name] = filtered_session_trigger_df
    
    print(f"  原始trials: {len(session_trigger_df)}, 过滤后trials: {len(filtered_session_trigger_df)}, 去除outliers: {len(outlier_orders)}")

print(f"\n{'='*60}")
print("Step 1完成: Correlation计算和outlier识别完成")
print(f"{'='*60}")

# ============================================================================
# Step 2: 使用过滤后的trigger_df计算PSTH
# ============================================================================
print(f"\n{'='*60}")
print("Step 2: 计算PSTH（使用过滤后的trigger_df）")
print(f"{'='*60}")

# 为每个session生成PSTH
all_session_psth = {}

for session_name in target_month_session_names:
    print(f"\n{'='*60}")
    print(f"处理session: {session_name}")
    print(f"{'='*60}")
    
    # 使用过滤后的trigger_df
    if session_name not in filtered_trigger_df_dict:
        print(f"  警告: 未找到过滤后的trigger_df，跳过")
        continue
    
    session_trigger_df = filtered_trigger_df_dict[session_name]
    
    if len(session_trigger_df) == 0:
        print(f"  警告: 过滤后的trigger_df为空，跳过")
        continue
    
    print(f"  使用过滤后的trigger_df: {len(session_trigger_df)} 个trials")
    
    # 读取该session的neuron信息
    session_dir = os.path.join(CLIQUE_DIR, session_name)
    session_neuron_inf_path = os.path.join(session_dir, "neuron_inf.pickle")
    
    if not os.path.exists(session_neuron_inf_path):
        print(f"  警告: 未找到neuron_inf.pickle: {session_neuron_inf_path}，跳过")
        continue
    
    with open(session_neuron_inf_path, 'rb') as f:
        session_neuron_inf = pickle.load(f)
    
    # 读取该session的spike数据
    session_spike_data = {}
    session_gt_detect_path = os.path.join(session_dir, "gt_detect_array.csv")
    if os.path.exists(session_gt_detect_path):
        session_gt_detect_df = pd.read_csv(session_gt_detect_path)
        session_gt_detect_df['time'] = pd.to_numeric(session_gt_detect_df['time'], errors='coerce')
        session_gt_detect_df['unit_id'] = pd.to_numeric(session_gt_detect_df['unit_id'], errors='coerce')
        session_gt_detect_df = session_gt_detect_df.dropna(subset=['time', 'unit_id'])
        
        for neuron_id in session_neuron_inf.keys():
            neuron_spikes = session_gt_detect_df[session_gt_detect_df['unit_id'] == neuron_id]['time'].values
            if len(neuron_spikes) > 0:
                session_spike_data[neuron_id] = neuron_spikes
    
    # 准备trigger信息
    # 注意：trigger_df中的start和end已经是10kHz采样率
    session_trigger_df = session_trigger_df.reset_index(drop=True)
    session_trigger_df['start_extended'] = session_trigger_df['start'] - int(EXTEND_TIME * SAMPLING_RATE)
    session_trigger_df['end_extended'] = session_trigger_df['start'] + int((STIMULUS_DURATION + EXTEND_TIME) * SAMPLING_RATE)
    
    n_trials = len(session_trigger_df)
    
    # 初始化PSTH矩阵: (n_trial, n_time_bins, n_neuron)
    psth_matrix = np.zeros((n_trials, n_time_bins, n_neurons))
    trial_image_id = []
    
    print(f"  开始计算PSTH...")
    print(f"  矩阵形状: ({n_trials}, {n_time_bins}, {n_neurons})")
    
    # 遍历所有trials
    for trial_idx, (_, trial) in enumerate(session_trigger_df.iterrows()):
        if trial_idx % 50 == 0:
            print(f"    处理trial {trial_idx}/{n_trials}")
        
        # 获取trial的image信息
        image_id = trial.get('image', 'unknown')
        trial_image_id.append(image_id)
        
        start_ext = int(trial['start_extended'])
        end_ext = int(trial['end_extended'])
        
        # 遍历所有基准neurons
        for neuron_idx, neuron_id in enumerate(baseline_neuron_ids):
            # 检查该neuron在当前session是否存在
            if neuron_id in session_spike_data:
                neuron_spikes = session_spike_data[neuron_id]
                
                # 获取该trial内的spikes
                trial_spikes = neuron_spikes[(neuron_spikes >= start_ext) & (neuron_spikes <= end_ext)]
                
                if len(trial_spikes) > 0:
                    # 转换为相对时间（秒）
                    relative_spikes = (trial_spikes - start_ext) / SAMPLING_RATE
                    
                    # 创建SpikeTrain对象
                    spiketrain = neo.SpikeTrain(
                        relative_spikes * 1000 * ms, 
                        t_stop=total_time_extended * 1000 * ms, 
                        t_start=0 * ms
                    )
                    
                    # 计算instantaneous rate
                    inst_rate = instantaneous_rate(spiketrain, kernel=gk, sampling_period=bin_size_ms * ms)
                    psth_trial = inst_rate.magnitude.flatten()
                else:
                    psth_trial = np.zeros(n_time_bins)
            else:
                # 如果neuron缺失，置零
                psth_trial = np.zeros(n_time_bins)
            
            # 确保长度一致
            if len(psth_trial) < n_time_bins:
                psth_trial = np.pad(psth_trial, (0, n_time_bins - len(psth_trial)), 'constant')
            elif len(psth_trial) > n_time_bins:
                psth_trial = psth_trial[:n_time_bins]
            
            # 存储到矩阵中
            psth_matrix[trial_idx, :, neuron_idx] = psth_trial
    
    print(f"  完成! PSTH矩阵形状: {psth_matrix.shape}")
    
    # 保存结果
    session_output_dir = os.path.join(OUTPUT_DIR, session_name)
    os.makedirs(session_output_dir, exist_ok=True)
    
    psth_output_path = os.path.join(session_output_dir, "psth_matrix.npy")
    trial_image_output_path = os.path.join(session_output_dir, "trial_image_id.pkl")
    
    np.save(psth_output_path, psth_matrix)
    with open(trial_image_output_path, 'wb') as f:
        pickle.dump(trial_image_id, f)
    
    all_session_psth[session_name] = {
        'psth_matrix': psth_matrix,
        'trial_image_id': trial_image_id,
        'n_trials': n_trials,
        'n_neurons': n_neurons
    }
    
    print(f"  已保存:")
    print(f"    PSTH矩阵: {psth_output_path}")
    print(f"    Trial image ID: {trial_image_output_path}")

print(f"\n{'='*60}")
print("所有session的PSTH生成完成!")
print(f"{'='*60}")

PSTH参数:
  Total time: 1.5 s
  Bin size: 50 ms
  Number of time bins: 30

读取基准session: mouse6_021322_natural_image_001
基准session有 31 个neurons

Step 1: 计算correlation和识别outlier
  原始trials: 1755, 过滤后trials: 1369, 去除outliers: 386
  原始trials: 1170, 过滤后trials: 1166, 去除outliers: 4
  原始trials: 1170, 过滤后trials: 1152, 去除outliers: 18
  原始trials: 1755, 过滤后trials: 1702, 去除outliers: 53
  原始trials: 1170, 过滤后trials: 1142, 去除outliers: 28
  原始trials: 1170, 过滤后trials: 1100, 去除outliers: 70
  原始trials: 1170, 过滤后trials: 1140, 去除outliers: 30
  原始trials: 1170, 过滤后trials: 1135, 去除outliers: 35
  原始trials: 1170, 过滤后trials: 1148, 去除outliers: 22
  原始trials: 1053, 过滤后trials: 1040, 去除outliers: 13
  原始trials: 1170, 过滤后trials: 1118, 去除outliers: 52
  原始trials: 1170, 过滤后trials: 1056, 去除outliers: 114

Step 1完成: Correlation计算和outlier识别完成

Step 2: 计算PSTH（使用过滤后的trigger_df）

处理session: mouse6_021322_natural_image_001
  使用过滤后的trigger_df: 1369 个trials
  开始计算PSTH...
  矩阵形状: (1369, 30, 31)
    处理trial 0/1369
    处理trial 50/1369
 

In [49]:
# ============================================================================
# 分类网络训练代码
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import math

# 定义TemporalEPEncoder（时间序列编码器）
class TemporalEPEncoder(nn.Module):
    """
    时间序列EP编码器（使用Conv1D替代Transformer）
    输入: (B, time_bins, neurons)
    输出: tokens (B, n_token, d_model)
    """
    def __init__(self, input_dim=31, time_bins=30, d_model=32, n_token=128, 
                 num_conv_layers=2, dropout=0.2, Cvae=32):
        super().__init__()
        self.input_dim = input_dim
        self.time_bins = time_bins
        self.d_model = d_model
        self.n_token = n_token
        self.Cvae = Cvae
        
        # 输入投影：对神经元维度降维
        if input_dim > 200:
            hidden_dim = min(input_dim // 4, d_model * 8)
            self.input_proj = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, d_model * 4),
                nn.LayerNorm(d_model * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
            )
        else:
            self.input_proj = nn.Sequential(
                nn.Linear(input_dim, d_model * 4),
                nn.LayerNorm(d_model * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
            )
        
        # 时间维度的1D卷积
        conv_layers = []
        for i in range(num_conv_layers):
            if i == 0:
                in_channels = d_model
            else:
                in_channels = d_model * 2
            
            if i == num_conv_layers - 1:
                out_channels = d_model
            else:
                out_channels = d_model * 2
            
            conv_layers.extend([
                nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_channels),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
        self.temporal_conv = nn.Sequential(*conv_layers)
        
        # 自适应池化到固定长度
        self.adaptive_pool = nn.AdaptiveAvgPool1d(n_token)
        
        # 最终投影层
        self.final_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        
        # Token投影到Cvae维度
        self.token_to_cvae = nn.Sequential(
            nn.Linear(d_model, Cvae),
            nn.LayerNorm(Cvae)
        )
        
        # 位置编码
        self.pos_embed = nn.Parameter(torch.zeros(1, n_token, d_model))
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化模型权重"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    
    def forward(self, x, return_condition_vector=False):
        """
        前向传播
        Args:
            x: (B, time_bins, input_dim)
            return_condition_vector: 是否返回条件向量（这里不使用）
        Returns:
            tokens: (B, n_token, d_model)
        """
        B = x.shape[0]
        
        # 检查输入
        if torch.isnan(x).any() or torch.isinf(x).any():
            x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)
        
        # 1. 输入投影
        x = self.input_proj(x)  # (B, time_bins, d_model)
        
        # 2. 转换为卷积输入格式
        x = x.transpose(1, 2)  # (B, d_model, time_bins)
        
        # 3. 时间维度的1D卷积
        x = self.temporal_conv(x)  # (B, d_model, time_bins)
        
        # 4. 自适应池化到n_token长度
        x = self.adaptive_pool(x)  # (B, d_model, n_token)
        
        # 5. 转回 (B, n_token, d_model)
        x = x.transpose(1, 2)  # (B, n_token, d_model)
        
        # 6. 最终投影
        x = self.final_proj(x)  # (B, n_token, d_model)
        
        # 7. 添加位置编码
        x = x + self.pos_embed  # (B, n_token, d_model)
        
        # 8. 投影到Cvae维度，生成tokens
        tokens = self.token_to_cvae(x)  # (B, n_token, Cvae)
        
        return tokens

# 定义分类模型（基于TemporalEPEncoder）
class ClassificationModel(nn.Module):
    """
    分类模型：使用TemporalEPEncoder作为特征提取器，添加分类头
    输入: (B, time_bins, neurons)
    输出: (B, num_classes) - 分类logits
    """
    def __init__(self, input_dim, time_bins, num_classes, d_model=32, n_token=128, 
                 num_conv_layers=2, dropout=0.2, hidden_dim=256):
        super().__init__()
        
        # 特征提取器（TemporalEPEncoder）
        self.encoder = TemporalEPEncoder(
            input_dim=input_dim,
            time_bins=time_bins,
            d_model=d_model,
            n_token=n_token,
            num_conv_layers=num_conv_layers,
            dropout=dropout,
            Cvae=d_model
        )
        
        # 分类头：从token特征到类别
        # 使用token序列的平均池化作为全局特征
        self.classifier = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x):
        """
        前向传播
        Args:
            x: (B, time_bins, input_dim) - PSTH数据
        Returns:
            logits: (B, num_classes) - 分类logits
        """
        # 获取encoder的中间特征（在投影到Cvae之前）
        # 我们需要修改encoder来返回中间特征，或者直接使用tokens的平均值
        tokens = self.encoder(x, return_condition_vector=False)  # (B, n_token, Cvae)
        
        # 对token序列进行平均池化，得到全局特征
        # 注意：tokens的维度是(B, n_token, Cvae)，其中Cvae=d_model
        global_feature = tokens.mean(dim=1)  # (B, Cvae) = (B, d_model)
        
        # 分类
        logits = self.classifier(global_feature)  # (B, num_classes)
        
        return logits

# 定义数据集
class PSTHDataset(Dataset):
    def __init__(self, psth_data, labels):
        """
        Args:
            psth_data: (n_trials, time_bins, n_neurons) - PSTH矩阵
            labels: (n_trials,) - 图像ID标签
        """
        self.psth_data = torch.tensor(psth_data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.psth_data)
    
    def __getitem__(self, idx):
        return self.psth_data[idx], self.labels[idx]

# 训练函数
def train_classification_model(model, train_loader, val_loader, num_epochs=50, 
                               lr=1e-3, device='cuda', save_path=None):
    """
    训练分类模型
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    best_val_acc = 0.0
    train_losses = []
    val_accs = []
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]', leave=False)
        for psth_data, labels in train_pbar:
            psth_data = psth_data.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            logits = model(psth_data)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100*train_correct/train_total:.2f}%'
            })
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100 * train_correct / train_total
        train_losses.append(avg_train_loss)
        
        # 验证阶段
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]', leave=False)
            for psth_data, labels in val_pbar:
                psth_data = psth_data.to(device)
                labels = labels.to(device)
                
                logits = model(psth_data)
                _, predicted = torch.max(logits.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'acc': f'{100*val_correct/val_total:.2f}%'
                })
        
        val_acc = 100 * val_correct / val_total
        val_accs.append(val_acc)
        
        scheduler.step()
        
        print(f'Epoch {epoch+1}/{num_epochs}: Train Loss={avg_train_loss:.4f}, '
              f'Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%')
        
        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            if save_path:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                }, save_path)
                print(f'  -> 保存最佳模型 (Val Acc: {val_acc:.2f}%)')
    
    return train_losses, val_accs, best_val_acc

# ============================================================================
# 方式1：使用所有月份的数据，统一进行训练和测试
# ============================================================================
print(f"\n{'='*60}")
print("方式1：使用所有月份的数据，统一进行训练和测试")
print(f"{'='*60}")

# 收集所有session的PSTH数据
all_psth_data = []
all_image_ids = []
all_session_names = []

for session_name in target_month_session_names:
    session_output_dir = os.path.join(OUTPUT_DIR, session_name)
    psth_path = os.path.join(session_output_dir, "psth_matrix.npy")
    image_id_path = os.path.join(session_output_dir, "trial_image_id.pkl")
    
    if os.path.exists(psth_path) and os.path.exists(image_id_path):
        psth_matrix = np.load(psth_path)  # (n_trials, time_bins, n_neurons)
        with open(image_id_path, 'rb') as f:
            trial_image_ids = pickle.load(f)
        
        all_psth_data.append(psth_matrix)
        all_image_ids.extend(trial_image_ids)
        all_session_names.extend([session_name] * len(trial_image_ids))
        print(f"  加载 {session_name}: {psth_matrix.shape[0]} trials")

# 合并所有数据
if len(all_psth_data) > 0:
    combined_psth = np.concatenate(all_psth_data, axis=0)  # (total_trials, time_bins, n_neurons)
    print(f"\n合并后数据形状: {combined_psth.shape}")
    print(f"总trials数: {len(all_image_ids)}")
    
    # 创建图像ID到类别索引的映射
    unique_image_ids = sorted(list(set(all_image_ids)))
    num_classes = len(unique_image_ids)
    image_id_to_class = {img_id: idx for idx, img_id in enumerate(unique_image_ids)}
    class_labels = np.array([image_id_to_class[img_id] for img_id in all_image_ids])
    
    print(f"唯一图像数量: {num_classes}")
    print(f"类别标签范围: {class_labels.min()} - {class_labels.max()}")
    
    # 划分训练集和测试集（80-20）
    train_indices, test_indices = train_test_split(
        np.arange(len(class_labels)), 
        test_size=0.2, 
        random_state=42, 
        stratify=class_labels
    )
    
    train_psth = combined_psth[train_indices]
    train_labels = class_labels[train_indices]
    test_psth = combined_psth[test_indices]
    test_labels = class_labels[test_indices]
    
    print(f"\n训练集: {len(train_indices)} trials")
    print(f"测试集: {len(test_indices)} trials")
    
    # 创建数据集和数据加载器
    train_dataset = PSTHDataset(train_psth, train_labels)
    test_dataset = PSTHDataset(test_psth, test_labels)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
    
    # 创建模型
    time_bins = combined_psth.shape[1]
    n_neurons = combined_psth.shape[2]
    
    model_all = ClassificationModel(
        input_dim=n_neurons,
        time_bins=time_bins,
        num_classes=num_classes,
        d_model=32,
        n_token=128,
        num_conv_layers=2,
        dropout=0.2,
        hidden_dim=256
    )
    
    print(f"\n模型参数数量: {sum(p.numel() for p in model_all.parameters()):,}")
    
    # 训练模型
    save_path_all = os.path.join(OUTPUT_DIR, "classification_model_all_months.pth")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\n使用设备: {device}")
    
    train_losses_all, val_accs_all, best_val_acc_all = train_classification_model(
        model=model_all,
        train_loader=train_loader,
        val_loader=test_loader,
        num_epochs=50,
        lr=1e-3,
        device=device,
        save_path=save_path_all
    )
    
    print(f"\n方式1训练完成！最佳验证准确率: {best_val_acc_all:.2f}%")
    print(f"模型已保存至: {save_path_all}")

# ============================================================================
# 方式2：滑动窗口训练（对于某个月份，使用前面所有月份的数据训练，该月份验证）
# ============================================================================
print(f"\n{'='*60}")
print("方式2：滑动窗口训练")
print(f"{'='*60}")

# 为每个session准备数据
session_data_dict = {}
for session_name in target_month_session_names:
    session_output_dir = os.path.join(OUTPUT_DIR, session_name)
    psth_path = os.path.join(session_output_dir, "psth_matrix.npy")
    image_id_path = os.path.join(session_output_dir, "trial_image_id.pkl")
    
    if os.path.exists(psth_path) and os.path.exists(image_id_path):
        psth_matrix = np.load(psth_path)
        with open(image_id_path, 'rb') as f:
            trial_image_ids = pickle.load(f)
        
        session_data_dict[session_name] = {
            'psth': psth_matrix,
            'image_ids': trial_image_ids
        }

# 获取所有唯一的图像ID（用于创建统一的类别映射）
all_unique_image_ids = set()
for session_name, data in session_data_dict.items():
    all_unique_image_ids.update(data['image_ids'])
all_unique_image_ids = sorted(list(all_unique_image_ids))
num_classes_slide = len(all_unique_image_ids)
image_id_to_class_slide = {img_id: idx for idx, img_id in enumerate(all_unique_image_ids)}

print(f"所有session的唯一图像数量: {num_classes_slide}")

# 滑动窗口训练（从第2个月开始，因为第1个月没有前面的数据）
slide_results = {}

for test_month_idx in range(1, len(target_month_session_names)):
    test_session = target_month_session_names[test_month_idx]
    train_sessions = target_month_session_names[:test_month_idx]
    
    print(f"\n{'='*60}")
    print(f"实验 {test_month_idx}/11: 测试月份 = {test_session}")
    print(f"训练月份: {', '.join(train_sessions)}")
    print(f"{'='*60}")
    
    # 收集训练数据
    train_psth_list = []
    train_labels_list = []
    
    for train_session in train_sessions:
        if train_session in session_data_dict:
            psth = session_data_dict[train_session]['psth']
            image_ids = session_data_dict[train_session]['image_ids']
            labels = np.array([image_id_to_class_slide[img_id] for img_id in image_ids])
            
            train_psth_list.append(psth)
            train_labels_list.extend(labels)
    
    if len(train_psth_list) == 0:
        print(f"  警告: 没有训练数据，跳过")
        continue
    
    train_psth_combined = np.concatenate(train_psth_list, axis=0)
    train_labels_combined = np.array(train_labels_list)
    
    # 测试数据
    if test_session not in session_data_dict:
        print(f"  警告: 测试session数据不存在，跳过")
        continue
    
    test_psth = session_data_dict[test_session]['psth']
    test_image_ids = session_data_dict[test_session]['image_ids']
    test_labels = np.array([image_id_to_class_slide[img_id] for img_id in test_image_ids])
    
    print(f"  训练集: {train_psth_combined.shape[0]} trials")
    print(f"  测试集: {test_psth.shape[0]} trials")
    
    # 创建数据集和数据加载器
    train_dataset_slide = PSTHDataset(train_psth_combined, train_labels_combined)
    test_dataset_slide = PSTHDataset(test_psth, test_labels)
    
    train_loader_slide = DataLoader(train_dataset_slide, batch_size=32, shuffle=True, num_workers=4)
    test_loader_slide = DataLoader(test_dataset_slide, batch_size=32, shuffle=False, num_workers=4)
    
    # 创建模型
    time_bins = train_psth_combined.shape[1]
    n_neurons = train_psth_combined.shape[2]
    
    model_slide = ClassificationModel(
        input_dim=n_neurons,
        time_bins=time_bins,
        num_classes=num_classes_slide,
        d_model=32,
        n_token=128,
        num_conv_layers=2,
        dropout=0.2,
        hidden_dim=256
    )
    
    # 训练模型
    save_path_slide = os.path.join(OUTPUT_DIR, f"classification_model_slide_month_{test_month_idx}.pth")
    
    train_losses_slide, val_accs_slide, best_val_acc_slide = train_classification_model(
        model=model_slide,
        train_loader=train_loader_slide,
        val_loader=test_loader_slide,
        num_epochs=50,
        lr=1e-3,
        device=device,
        save_path=save_path_slide
    )
    
    slide_results[test_month_idx] = {
        'test_session': test_session,
        'train_sessions': train_sessions,
        'best_val_acc': best_val_acc_slide,
        'train_losses': train_losses_slide,
        'val_accs': val_accs_slide
    }
    
    print(f"  实验 {test_month_idx}/11 完成！最佳验证准确率: {best_val_acc_slide:.2f}%")
    print(f"  模型已保存至: {save_path_slide}")

# 保存滑动窗口训练结果汇总
print(f"\n{'='*60}")
print("滑动窗口训练结果汇总")
print(f"{'='*60}")

slide_summary_path = os.path.join(OUTPUT_DIR, "slide_training_summary.pkl")
with open(slide_summary_path, 'wb') as f:
    pickle.dump(slide_results, f)

print(f"\n滑动窗口训练结果已保存至: {slide_summary_path}")
print(f"\n各月份验证准确率:")
for month_idx, result in slide_results.items():
    print(f"  月份 {month_idx} ({result['test_session']}): {result['best_val_acc']:.2f}%")

print(f"\n{'='*60}")
print("所有训练完成！")
print(f"{'='*60}")



方式1：使用所有月份的数据，统一进行训练和测试
  加载 mouse6_021322_natural_image_001: 1369 trials
  加载 mouse6_022522_natural_image_001: 1166 trials
  加载 mouse6_031722_natural_image_001: 1152 trials
  加载 mouse6_042422_natural_image_001: 1702 trials
  加载 mouse6_052422_natural_image_001: 1142 trials
  加载 mouse6_062422_natural_image_001: 1100 trials
  加载 mouse6_072322_natural_image_001: 1140 trials
  加载 mouse6_082322_natural_image_001: 1135 trials
  加载 mouse6_092422_natural_image_001: 1148 trials
  加载 mouse6_102122_natural_image_001: 1040 trials
  加载 mouse6_112022_natural_image_001: 1118 trials
  加载 mouse6_122022_natural_image_001: 1056 trials

合并后数据形状: (14268, 30, 31)
总trials数: 14268
唯一图像数量: 117
类别标签范围: 0 - 116

训练集: 11414 trials
测试集: 2854 trials

模型参数数量: 84,597

使用设备: cuda


Epoch 1/50: Train Loss=4.3449, Train Acc=3.87%, Val Acc=7.25%
  -> 保存最佳模型 (Val Acc: 7.25%)


Epoch 2/50: Train Loss=3.8920, Train Acc=8.29%, Val Acc=10.48%
  -> 保存最佳模型 (Val Acc: 10.48%)


Epoch 3/50: Train Loss=3.6887, Train Acc=11.05%, Val Acc=12.58%
  -> 保存最佳模型 (Val Acc: 12.58%)


Epoch 4/50: Train Loss=3.5442, Train Acc=12.88%, Val Acc=15.45%
  -> 保存最佳模型 (Val Acc: 15.45%)


...

Epoch 46/50: Train Loss=2.1996, Train Acc=38.59%, Val Acc=27.18%


Epoch 47/50: Train Loss=2.1843, Train Acc=39.39%, Val Acc=26.80%


Epoch 48/50: Train Loss=2.1802, Train Acc=39.25%, Val Acc=26.89%


Epoch 49/50: Train Loss=2.1775, Train Acc=39.46%, Val Acc=27.75%


Epoch 50/50: Train Loss=2.1686, Train Acc=39.30%, Val Acc=27.56%
  实验 11/11 完成！最佳验证准确率: 27.84%
  模型已保存至: /media/ubuntu/sda/mouse_test/processed_results/psth_results/classification_model_slide_month_11.pth

滑动窗口训练结果汇总

滑动窗口训练结果已保存至: /media/ubuntu/sda/mouse_test/processed_results/psth_results/slide_training_summary.pkl

各月份验证准确率:
  月份 1 (mouse6_022522_natural_image_001): 1.89%
  月份 2 (mouse6_031722_natural_image_001): 12.07%
  月份 3 (mouse6_042422_natural_image_001): 4.41%
  月份 4 (mouse6_052422_natural_image_001): 17.25%
  月份 5 (mouse6_062422_natural_image_001): 12.73%
  月份 6 (mouse6_072322_natural_image_001): 18.68%
  月份 7 (mouse6_082322_natural_image_001): 23.52%
  月份 8 (mouse6_092422_natural_image_001): 31.10%
  月份 9 (mouse6_102122_natural_image_001): 16.73%
  月份 10 (mouse6_112022_natural_image_001): 24.06%
  月份 11 (mouse6_122022_natural_image_001): 27.84%

所有训练完成！
